# AnyCam — 05 · Step 7: SVCT — one model for any camera

Three models, all trained ONLY on random virtual cameras (the 7 real camera profiles are held out):
1. **SVCT (ours)**: wavelength-aware first layer + two cameras per batch + consistency loss + calibration errors
2. **Ablation: no wavelength layer** — same random cameras, bands interpolated back to 204, ordinary ResNet
3. **Ablation: no consistency loss** — SVCT with one camera per batch

Each is tested on the 7 real cameras (ideal / drift / calibration+noise / both), on the full 204 bands and on
20 random unseen cameras. Total ~3-4 h. Same setup: **HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**.
Best: **Save Version → Save & Run All**. Then save to GitHub as `kaggle_runs/05_step7_svct.ipynb`.

In [ ]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

In [ ]:
# 1) SVCT — our method
!python experiments/train_svct.py --data {DATA} --stem hyper --views 2 --cons 1.0 --radio 0.5 --out runs/svct_main

In [ ]:
# 2) ablation: random virtual cameras but NO wavelength-aware layer (interpolate to 204 bands)
!python experiments/train_svct.py --data {DATA} --stem interp --views 2 --cons 1.0 --radio 0.5 --out runs/svct_interp

In [ ]:
# 3) ablation: SVCT without the consistency loss (one camera per batch)
!python experiments/train_svct.py --data {DATA} --stem hyper --views 1 --radio 0.5 --out runs/svct_nocons

In [ ]:
# comparison with Step 6 (204-band model, one model per camera)
!python scripts/plot_step7.py --baseline results/step06_camera_test/camera_eval.csv --runs "SVCT (ours)=runs/svct_main" "no wavelength layer=runs/svct_interp" "no consistency=runs/svct_nocons" --out runs/step7_compare
from IPython.display import Image, display
display(Image('runs/step7_compare/step7_compare.png'))
!for f in runs/svct_*/camera_eval.csv; do echo "### $f"; cat $f; done

## Done
When every cell has finished: **Save Version → Quick Save** with GitHub ticked → file `kaggle_runs/05_step7_svct.ipynb`. Tell Claude "done".